# Hinge loss SVM

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = "FeatureC_Repeated"
OUTPUT_PATH = "SVM_FeatureC_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

# SVM hyperparameter candidates
# alpha is the regularization strength in SGDClassifier
ALPHA_VALUES = [1e-4, 1e-3, 1e-2]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [3]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [4]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)
    train_indices = []
    test_indices = []
    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)
        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [5]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [6]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [7]:
def train_and_evaluate_hinge_svm(train_df, test_df, alpha_value):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    svm_model = SGDClassifier(
        loss="hinge",
        penalty="l2",
        alpha=alpha_value,
        max_iter=1000,
        tol=1e-3,
        random_state=42
    )

    svm_model.fit(X_train_scaled, y_train)

    y_pred = svm_model.predict(X_test_scaled)

    # For hinge-loss SVM, decision_function gives the margin score
    y_score = svm_model.decision_function(X_test_scaled)

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_score)

    return metrics

In [8]:
def tune_svm_alpha_from_scratch(
    train_df,
    alpha_values,
    n_inner_repeats=3,
    valid_ratio=0.2,
    base_seed=100
):
    tuning_records = []

    for alpha_value in alpha_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_hinge_svm(
                train_df=inner_train_df,
                test_df=valid_df,
                alpha_value=alpha_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "alpha": alpha_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_alpha = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["alpha"]

    return best_alpha, tuning_df

In [9]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_C_train.csv")
    test_path = os.path.join(repeat_folder, "feature_C_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_alpha, tuning_df = tune_svm_alpha_from_scratch(
        train_df=train_df,
        alpha_values=ALPHA_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=2000 + repeat_id * 10
    )

    print("Best alpha:", best_alpha)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_hinge_svm(
        train_df=train_df,
        test_df=test_df,
        alpha_value=best_alpha
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_alpha": best_alpha,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6409
Precision: 0.6478
Recall   : 0.6169
F1       : 0.632
AUC      : 0.6795
Outer Repeat 02
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6389
Precision: 0.643
Recall   : 0.6239
F1       : 0.6333
AUC      : 0.6792
Outer Repeat 03
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6371
Precision: 0.6406
Recall   : 0.6239
F1       : 0.6322
AUC      : 0.6765
Outer Repeat 04
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6386
Precision: 0.6455
Recall   : 0.6142
F1       : 0.6295
AUC      : 0.6812
Outer Repeat 05
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6381
Precision: 0.6441
Recall   : 0.6164
F1       : 0.6299
AUC      : 0.6794
Outer Repeat 06
Train shape: (160000, 253)
Test shape: (39999, 253)
Best alpha: 0.001
Accuracy : 0.6421
Precision: 0.6455


In [10]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "SVM_FeatureC_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "SVM_FeatureC_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
SVM_FeatureC_Results/SVM_FeatureC_repeated_results.csv
Saved tuning results to:
SVM_FeatureC_Results/SVM_FeatureC_tuning_results.csv


,outer_repeat,best_alpha,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,0.001,0.640941,0.647841,0.616858,0.631970,12331,13306,6703,7659,0.679465
1,2,0.001,0.638916,0.643001,0.623862,0.633287,12471,13085,6924,7519,0.679240
2,3,0.001,0.637141,0.640641,0.623912,0.632166,12472,13013,6996,7518,0.676521
3,4,0.001,0.638616,0.645497,0.614207,0.629463,12278,13266,6743,7712,0.681156
4,5,0.001,0.638066,0.644101,0.616358,0.629924,12321,13201,6808,7669,0.679431
5,6,0.001,0.642091,0.645547,0.629465,0.637404,12583,13100,6909,7407,0.681964
6,7,0.001,0.639016,0.644459,0.619410,0.631686,12382,13178,6831,7608,0.679519
7,8,0.001,0.639366,0.641740,0.630215,0.635925,12598,12976,7033,7392,0.679109
8,9,0.001,0.641116,0.646508,0.621961,0.633997,12433,13211,6798,7557,0.682320
9,10,0.001,0.642216,0.645518,0.630115,0.637724,12596,13092,6917,7394,0.681402


In [11]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "SVM_FeatureC_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.639748,0.001737,0.000549
1,precision,0.644485,0.002192,0.000693
2,recall,0.622636,0.005943,0.001879
3,f1,0.633355,0.002897,0.000916
4,auc,0.680013,0.001727,0.000546


In [12]:
best_alpha_frequency = results_df["best_alpha"].value_counts().reset_index()
best_alpha_frequency.columns = ["alpha", "frequency"]

best_alpha_frequency_path = os.path.join(
    OUTPUT_PATH,
    "SVM_FeatureC_best_alpha_frequency.csv"
)
best_alpha_frequency.to_csv(best_alpha_frequency_path, index=False, encoding="utf-8-sig")

best_alpha_frequency

,alpha,frequency
0,0.001,10
